In [1]:
import os
import csv
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import gc
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import joblib

In [2]:
print(torch.cuda.is_available())

True


In [3]:
def merge_phase_files(data_path, splits=["train", "valid"], phases=["I", "II", "III"], output_file="all_phases_ctod.csv"):
    merged = []

    for split in splits:
        for phase in phases:
            file = f"phase_{phase}_{split}.csv"
            path = os.path.join(data_path, file)
            if os.path.exists(path):
                df = pd.read_csv(path)
                merged.append(df)
                print(f" Loaded {file} with {len(df)} rows")
            else:
                print(f" File not found: {file}")

    if merged:
        all_data = pd.concat(merged, ignore_index=True)
        all_data.to_csv(os.path.join(data_path, output_file), index=False)
        print(f"\n Merged dataset saved as: {output_file} | Total rows: {len(all_data)}")
        return os.path.join(data_path, output_file)
    else:
        print("No files found to merge.")
        return None

In [4]:
merged_file = merge_phase_files(data_path="data/labelling/labelling_reduced")

 Loaded phase_I_train.csv with 4016 rows
 Loaded phase_II_train.csv with 5390 rows
 Loaded phase_III_train.csv with 3572 rows
 Loaded phase_I_valid.csv with 1006 rows
 Loaded phase_II_valid.csv with 1348 rows
 Loaded phase_III_valid.csv with 893 rows

 Merged dataset saved as: all_phases_ctod.csv | Total rows: 16225


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")
model = AutoModel.from_pretrained("medicalai/ClinicalBERT").to(device)
model.eval()

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Carol\.cache\huggingface\hub\models--medicalai--ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/542M [00:00<?, ?B/s]

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): 

In [6]:
def clean_protocol(text):
    if isinstance(text, str):
        lines = text.lower().split("~")
    else:
        return []
    lines = [line.strip() for line in lines if len(line.strip()) > 0]
    return lines

def split_protocol(protocol):
    lines = clean_protocol(protocol)
    inclusion = [l for l in lines if "inclusion" not in l and "exclusion" not in l and "subject" in l]
    exclusion = [l for l in lines if "inclusion" not in l and "exclusion" not in l and "subject" not in l]
    return inclusion, exclusion

def collect_cleaned_sentence_set(input_file):
    df = pd.read_csv(input_file)
    all_sentences = set()
    for protocol in df["criteria"].dropna():
        inclusion, exclusion = split_protocol(protocol)
        all_sentences.update(inclusion + exclusion)
    return all_sentences

In [7]:
class SentenceDataset(Dataset):
    def __init__(self, sentences):
        self.sentences = list(sentences)
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, idx):
        return self.sentences[idx]

# GPU-optimized batch embedding
def get_batched_embeddings(sentences, batch_size=64):
    dataset = SentenceDataset(sentences)
    dataloader = DataLoader(dataset, batch_size=batch_size)

    embeddings = []
    for batch in tqdm(dataloader, desc="Embedding in batches"):
        inputs = tokenizer(list(batch), return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
        cls_batch = outputs.last_hidden_state[:, 0, :].cpu()
        embeddings.extend(cls_batch)
        del inputs, outputs, cls_batch
        torch.cuda.empty_cache()
        gc.collect()
    return embeddings

In [8]:
def save_sentence_bert_dict_batched(sentences, output_file, batch_size=64, save_every=10000):
    if os.path.exists(output_file):
        sentence2embedding = joblib.load(output_file)
        sentence2embedding = {k: torch.tensor(v) for k, v in sentence2embedding.items()}
        print(f"[Resuming] Loaded existing embeddings: {len(sentence2embedding)}")
    else:
        sentence2embedding = {}

    sentences = list(set(sentences))
    remaining = [s for s in sentences if s not in sentence2embedding]
    print(f"[Embedding] Remaining sentences: {len(remaining)}")

    for i in range(0, len(remaining), batch_size):
        batch = remaining[i:i + batch_size]
        try:
            batch_embeddings = get_batched_embeddings(batch, batch_size)
            for s, e in zip(batch, batch_embeddings):
                sentence2embedding[s] = e
        except Exception as e:
            print(f"[!] Batch error: {e}")
            continue

        if (i // batch_size) % (save_every // batch_size) == 0 or i + batch_size >= len(remaining):
            numpy_dict = {k: v.numpy() for k, v in sentence2embedding.items()}
            joblib.dump(numpy_dict, output_file)
            print(f"Intermediate save: {len(numpy_dict)} entries")
            del numpy_dict
            gc.collect()

    print(f"Final save: {output_file} | Total embeddings: {len(sentence2embedding)}")
    return sentence2embedding

In [9]:
def protocol2feature(protocol, sentence2vec):
    inclusion, exclusion = split_protocol(protocol)
    inc_vecs = [sentence2vec[s] for s in inclusion if s in sentence2vec]
    exc_vecs = [sentence2vec[s] for s in exclusion if s in sentence2vec]

    if not inc_vecs:
        inc = torch.zeros(1, 768)
    else:
        inc = torch.stack(inc_vecs).mean(0, keepdim=True)

    if not exc_vecs:
        exc = torch.zeros(1, 768)
    else:
        exc = torch.stack(exc_vecs).mean(0, keepdim=True)

    return torch.cat([inc, exc], dim=1)

def prepare_criteria_feature(data_path, embedding_path, output_path="criteria"):
    os.makedirs(os.path.join(data_path, output_path), exist_ok=True)
    sentence2vec = joblib.load(os.path.join(data_path, embedding_path))
    sentence2vec = {k: torch.tensor(v) for k, v in sentence2vec.items()}

    for phase in tqdm(["I", "II", "III"], desc="Phases"):
        for split in ["train", "valid"]:
            file = f"phase_{phase}_{split}.csv"
            filepath = os.path.join(data_path, file)

            if not os.path.exists(filepath):
                print(f"File not found: {filepath}")
                continue

            df = pd.read_csv(filepath)
            features = []

            for _, row in df.iterrows():
                protocol = row.get("criteria", "")
                try:
                    feat = protocol2feature(protocol, sentence2vec)
                except Exception as e:
                    print(f"Skipping protocol: {e}")
                    feat = torch.zeros(1, 1536)
                features.append(feat)

            stacked = torch.cat(features, dim=0).numpy()
            out_file = os.path.join(data_path, output_path, f"phase_{phase}_{split}.npy")
            np.save(out_file, stacked)
            print(f"Saved: {out_file} | Shape: {stacked.shape}")

In [10]:
def main():
    input_file = "data/labelling/labelling_reduced/all_phases_ctod.csv"
    output_pickle = "sentence2embedding.pkl"

    # Step 1: Collect and embed
    sentences = collect_cleaned_sentence_set(input_file)
    print(f"Total unique cleaned sentences: {len(sentences)}")

    save_sentence_bert_dict_batched(
        sentences,
        output_file=os.path.join("data/labelling", output_pickle),
        batch_size=64,
        save_every=10000
    )

    # Step 2: Build .npy for each trial
    prepare_criteria_feature(data_path="data/labelling", embedding_path=output_pickle)

if __name__ == "__main__":
    main()

Total unique cleaned sentences: 276466
[Embedding] Remaining sentences: 276466


Embedding in batches: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.14it/s]


Intermediate save: 64 entries


Embedding in batches: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.73it/s]



model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

...

Saved: data/labelling\criteria\phase_I_valid.npy | Shape: (4019, 1536)
Saved: data/labelling\criteria\phase_II_train.npy | Shape: (21568, 1536)


Phases:  67%|██████████████████████████████████████████████████▋                         | 2/3 [00:09<00:04,  5.00s/it]

Saved: data/labelling\criteria\phase_II_valid.npy | Shape: (5392, 1536)
Saved: data/labelling\criteria\phase_III_train.npy | Shape: (14281, 1536)


Phases: 100%|████████████████████████████████████████████████████████████████████████████| 3/3 [00:12<00:00,  4.20s/it]

Saved: data/labelling\criteria\phase_III_valid.npy | Shape: (3571, 1536)


In [11]:
data = np.load("data/labelling/criteria/phase_I_train.npy")
data

array([[ 0.        ,  0.        ,  0.        , ...,  0.06107395,
        -0.01202773, -0.30792195],
       [ 0.        ,  0.        ,  0.        , ...,  0.06552131,
         0.01648149, -0.31821772],
       [ 0.        ,  0.        ,  0.        , ...,  0.03222304,
         0.05265201, -0.35248336],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.04023397,
         0.02660605, -0.26394317],
       [ 0.        ,  0.        ,  0.        , ...,  0.09540904,
        -0.00060552, -0.3122934 ],
       [ 0.        ,  0.        ,  0.        , ...,  0.0812205 ,
         0.00396986, -0.32995352]], dtype=float32)